# Advance RAG Chatbot - Empirical Retrieval Benchmark (GPU Accelerated)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/akhilsainigit/Advance_RAG_Chatbot/blob/main/notebooks/colab_rag_benchmark.ipynb)

This notebook runs the 35-case RAG retrieval benchmark comparing:
1. **Dense Vector Retrieval** (BGE-M3 + Qdrant)
2. **Sparse Keyword Retrieval** (BM25Okapi)
3. **Hybrid Fusion** (Dense + Sparse via RRF)
4. **Hybrid Fusion + Cross-Encoder Reranking** (`BAAI/bge-reranker-v2-m3` on CUDA GPU)

On GPU (T4 / A100), cross-encoder inference drops from **~21 seconds/query on CPU** to **~15 milliseconds/query**.

In [ ]:
# 1. Verify GPU acceleration
!nvidia-smi

In [ ]:
# 2. Clone repository & install dependencies
!git clone https://github.com/akhilsainigit/Advance_RAG_Chatbot.git
%cd Advance_RAG_Chatbot
!pip install -r requirements.txt

In [ ]:
# 3. Configure environment for embedded local evaluation (no external keys needed)
env_content = """
QDRANT_URL=""
BM25_STORE_PATH="bm25_store.pkl"
COLLECTION_NAME="multidoc_rag"
RERANKER_MODEL_NAME="BAAI/bge-reranker-v2-m3"
USE_RERANKER=true
CORS_ORIGINS=["http://localhost:3000","http://localhost:8000","http://localhost:5173"]
"""
with open(".env", "w") as f:
    f.write(env_content.strip())
print(".env configured for GPU evaluation.")

In [ ]:
# 4. Ingest the 16 travel policy documents into embedded Qdrant & BM25 store
!python ingest.py --data-dir data/txt --recreate

In [ ]:
# 5. Run live benchmark suite across all 4 retrieval pipelines
!python -m evals.rag.runner

In [ ]:
# 6. Download evaluation report JSON
from google.colab import files
files.download("evals/results/rag_report.json")